## 权重转换

In [1]:
# !pip install tensorflow==2.16.1
import json
import os
import sys
import asyncio
import argparse
from collections import defaultdict
import time
import re

os.environ["JAX_PLATFORMS"] = "cpu"

import torch
import numpy as np
import jax.numpy as jnp
import jax
import orbax
import orbax.checkpoint as ocp
from etils import epath
from jax.sharding import PartitionSpec as PS
from flax.traverse_util import flatten_dict, unflatten_dict
import base64
from einops import rearrange


def decode_base64(encoded_str):
    decoded_bytes = base64.b64decode(encoded_str)
    decoded_str = decoded_bytes.decode('utf-8')
    return decoded_str

def encode_base64(decoded_str):
    # decoded_str = "opt_state.mu.params.token_embedder.embedding"
    encoded_string = base64.b64encode(decoded_str.encode('utf-8')).decode('utf-8')
    return encoded_string


In [2]:
p = 'gs://newproject-1-llm_base_models_europe-west4/v3.5mini/DreamMiniXLE64T40728/v3.5mini_moe_params_shape.json'
p = epath.Path(p)
with p.open('r') as f:
    shapedtype = json.load(f)
    
# load
mesh_axes = ['data', 'stage', 'fsdp', 'fsdp_transpose', 'sequence', 'tensor', 'tensor_transpose', 'tensor_sequence', 'expert', 'autoregressive']
axes = [1] * len(mesh_axes)
axes[2] = 1
devices = np.asarray(jax.devices()).reshape(axes)
mesh = jax.sharding.Mesh(devices, mesh_axes)
sharding = jax.sharding.NamedSharding(mesh, PS()) # Sharding is None because we use cpu to load weights
weight_dtype = jnp.bfloat16 # set restore weights dtype, np.float32 or np.float16
abstract_unboxed_params = {}
for k, shape in shapedtype.items():
    if not isinstance(k, tuple):
        k = tuple(k.split('/'))
    print(k, shape)
    abstract_unboxed_params[k] = jax.ShapeDtypeStruct(shape=shape, dtype=weight_dtype, sharding=sharding)    
    
abstract_unboxed_params = unflatten_dict(abstract_unboxed_params)
checkpoint_dir = 'gs://newproject-1-llm_base_models_europe-west4/v3.5mini/DreamMiniXLE64T40728/checkpoints/160000/items'

ckpt = epath.Path(checkpoint_dir)
ckptr = ocp.PyTreeCheckpointer()
restore_args = ocp.checkpoint_utils.construct_restore_args(abstract_unboxed_params)
# 如果restored只是一个带有模型名字的字典，没有具体的value矩阵，可以检查下abstract_unboxed_params是不是多了或者漏了params这个key
restored = ckptr.restore(
  ckpt, item={'params': abstract_unboxed_params}, transforms={}, restore_args={'params': restore_args}
)

2025-08-27 15:03:24.205038: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756307004.218164  470688 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756307004.222072  470688 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756307004.233437  470688 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756307004.233452  470688 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756307004.233454  470688 computation_placer.cc:177] computation placer alr

('params', 'decoder', 'compose_0', 'mudd_postnorm_0', 'scale') [2048]
('params', 'decoder', 'compose_0', 'mudd_prenorm_0', 'scale') [2048]
('params', 'decoder', 'compose_1', 'mudd_postnorm_1', 'scale') [2048]
('params', 'decoder', 'compose_1', 'mudd_prenorm_1', 'scale') [2048]
('params', 'decoder', 'compose_10', 'mudd_postnorm_10', 'scale') [2048]
('params', 'decoder', 'compose_10', 'mudd_prenorm_10', 'scale') [2048]
('params', 'decoder', 'compose_11', 'mudd_postnorm_11', 'scale') [2048]
('params', 'decoder', 'compose_11', 'mudd_prenorm_11', 'scale') [2048]
('params', 'decoder', 'compose_12', 'mudd_postnorm_12', 'scale') [2048]
('params', 'decoder', 'compose_12', 'mudd_prenorm_12', 'scale') [2048]
('params', 'decoder', 'compose_13', 'mudd_postnorm_13', 'scale') [2048]
('params', 'decoder', 'compose_13', 'mudd_prenorm_13', 'scale') [2048]
('params', 'decoder', 'compose_14', 'mudd_postnorm_14', 'scale') [2048]
('params', 'decoder', 'compose_14', 'mudd_prenorm_14', 'scale') [2048]
('param

I0827 15:03:27.158508  471329 google_auth_provider.cc:181] Running on GCE, using service account 626151558586-compute@developer.gserviceaccount.com


In [3]:
# 参数转换

mudd_map = {
    'mudd_mlp': {
        'pre_dense_proj1_norm': 'norm.scale',
        'dynamic_dense_conn1/kernel': 'w1.weight', # T
        'dynamic_dense_conn2/kernel': 'w2.weight', # T+reshape
        'dense_proj2.bias': 'w2.bias', # T+reshape
    },
    'decoder_norm/scale': 'norm.weight',
    'logits_dense/kernel': 'output.weight', # T
    'params/params/token_embedder/embedding': 'tok_embeddings.weight',
    "layers": {
        'layers_0/sub_0/self_attention/query/kernel': 'layers.0.attention.wq.weight', # T + reshape
        'layers_0/sub_0/self_attention/key/kernel': 'layers.0.attention.wk.weight', # T + reshape
        'layers_0/sub_0/self_attention/value/kernel': 'layers.0.attention.wv.weight', # T + reshape
        'layers_0/sub_0/self_attention/out/kernel': 'layers.0.attention.wo.weight', # T + reshape
        'layers_0/sub_0/self_attention/qk_norm/q_norm/scale': 'layers.0.attention.q_norm.scale',
        'layers_0/sub_0/self_attention/qk_norm/k_norm/scale': 'layers.0.attention.k_norm.scale',
        'layers_0/sub_0/self_attention/kv_shift/kv_shift_proj_k/kernel': 'layers.0.attention.kv_shift.dw_proj_k',
        'layers_0/sub_0/self_attention/kv_shift/kv_shift_proj_v/kernel': 'layers.0.attention.kv_shift.dw_proj_v',
        # 'layers_0/sub_0/mlp/wi_0/kernel': 'layers.0.feed_forward.w1.weight', # T  dense
        # 'layers_0/sub_0/mlp/wi_1/kernel': 'layers.0.feed_forward.w3.weight', # T
        # 'layers_0/sub_0/mlp/wo/kernel': 'layers.0.feed_forward.w2.weight', # T
        'layers_0/sub_0/moe/gate/kernel': 'layers.0.moe.router.weight', # T # moe
        'layers_0/sub_0/moe/wi_0': 'layers.0.moe.w1',
        'layers_0/sub_0/moe/wi_1': 'layers.0.moe.w3',
        'layers_0/sub_0/moe/wo': 'layers.0.moe.w2',
        'layers_0/sub_0/post_self_attention_layer_norm/scale': 'layers.0.ffn_norm.weight',
        'layers_0/sub_0/mudd_qkvnorm/pre_self_attention_layer_norm_q/scale': 'layers.0.attention_norm.0.weight',
        'layers_0/sub_0/mudd_qkvnorm/pre_self_attention_layer_norm_k/scale': 'layers.0.attention_norm.1.weight',
        'layers_0/sub_0/mudd_qkvnorm/pre_self_attention_layer_norm_v/scale': 'layers.0.attention_norm.2.weight',
        'layers_0/sub_0/self_attention/attention_op/q_dyn_w_proj/dw1/kernel': 'layers.0.attention.dyn_w_proj.dw1',
        'layers_0/sub_0/self_attention/attention_op/q_dyn_w_proj/dd/kernel': 'layers.0.attention.dyn_w_proj.dd',
        # 'layers_0/sub_0/self_attention/attention_op/q_dyn_w_proj/dd_bias': 'layers.0.attention.dyn_w_proj.dd_bias', # desen have it bug moe remove it
        'layers_0/sub_0/self_attention/attention_op/q_dyn_w_proj/qkw': 'layers.0.attention.dyn_w_proj.qkw',
        'layers_0.sub_0.self_attention.attention_op.q_dyn_w_proj.dw1.kernel': 'layers.0.attention.dyn_w_proj.dw1',
        'layers_0/sub_0/self_attention/attention_op/q_dyn_w_proj/w1_bias': 'layers.0.attention.dyn_w_proj.w1_bias', # reshape
        'layers_0/sub_0/self_attention/attention_op/q_dyn_w_proj/w2_bias': 'layers.0.attention.dyn_w_proj.w2_bias', # reshape
    }
}

torch_params = {}
# new_tensor = None
for key, tensor in flatten_dict(restored).items():
    new_tensor = None
    key = '/'.join(key)
    print(key, tensor.shape, tensor.sum())
    if 'mudd_prenorm' in key:
        if 'decoder/mudd_prenorm/' in key:
            new_key = 'prenorm_emb.weight'
        else:
            ldx = int(re.findall('mudd_prenorm_(\d+)', key)[0])
            new_key = f'prenorm.{ldx}.weight'
        new_tensor = tensor
        
    elif 'mudd_postnorm' in key:
        ldx = int(re.findall('mudd_postnorm_(\d+)', key)[0])
        new_key = f'postnorm.{ldx}.weight'
        new_tensor = tensor
        
    elif 'mudd_mlp' in key:
        ldx = int(re.findall('layers_(\d+)', key)[0]) # diff dense
        for k, v in mudd_map['mudd_mlp'].items():
            if k in key:
                new_key = f'dynamic_dense.{ldx}.{v}'
                break
        if 'dynamic_dense_conn1' in k:
            new_tensor = tensor.T
        elif 'dynamic_dense_conn2' in k:
            new_tensor = rearrange(tensor, 'K C L -> (C L) K')
        elif 'dense_proj2.bias' in k:
            new_tensor = tensor.reshape(-1)
        else:
            new_tensor = tensor
        
    elif 'sub_0' in key:
        ldx = int(re.findall('layers_(\d+)', key)[0])
        map_dict = {}
        for k, v in mudd_map['layers'].items():
            k = 'params/params/decoder/' + k.replace('layers_0/', f'layers_{ldx}/')
            v = v.replace('layers.0', f'layers.{ldx}')
            map_dict[k] = v
        new_key = map_dict[key]
        if re.search('(query|key|value)/kernel', key):
            new_tensor = rearrange(tensor, 'D N H -> (N H) D')
        elif re.search('out/kernel', key):
            new_tensor = rearrange(tensor, 'N H D -> D (N H)')
        # elif re.search('mlp\/w', key):  # diff dense
        #     new_tensor = tensor.T
        elif re.search('moe/gate/kernel', key): # diff dense
            new_tensor = tensor.T
        elif re.search('q_dyn_w_proj\/w\d', key):
            new_tensor = rearrange(tensor, 'C I K -> 1 (C I) K')
        else:
           new_tensor = tensor
    else:
        key_ = key.replace('params/params/decoder/', '')
        new_key = mudd_map[key_]
        if 'logits_dense/kernel' in key:
            new_tensor = tensor.T
        else:
            new_tensor = tensor
        
    if new_tensor is not None:
        # new_tensor = torch.from_numpy(np.array(new_tensor)).to(torch.float32)
        new_tensor = torch.from_numpy(np.array(new_tensor).astype(np.float32)).to(torch.bfloat16)
        torch_params[new_key] = new_tensor

# torch.save(torch_params, 'torch_params.bin')


params/params/decoder/compose_0/mudd_postnorm_0/scale (2048,) 245
params/params/decoder/compose_0/mudd_prenorm_0/scale (2048,) 114
params/params/decoder/compose_1/mudd_postnorm_1/scale (2048,) 228
params/params/decoder/compose_1/mudd_prenorm_1/scale (2048,) 109.5
params/params/decoder/compose_10/mudd_postnorm_10/scale (2048,) 266
params/params/decoder/compose_10/mudd_prenorm_10/scale (2048,) 126
params/params/decoder/compose_11/mudd_postnorm_11/scale (2048,) 318
params/params/decoder/compose_11/mudd_prenorm_11/scale (2048,) 122
params/params/decoder/compose_12/mudd_postnorm_12/scale (2048,) 290
params/params/decoder/compose_12/mudd_prenorm_12/scale (2048,) 139
params/params/decoder/compose_13/mudd_postnorm_13/scale (2048,) 288
params/params/decoder/compose_13/mudd_prenorm_13/scale (2048,) 146
params/params/decoder/compose_14/mudd_postnorm_14/scale (2048,) 241
params/params/decoder/compose_14/mudd_prenorm_14/scale (2048,) 112.5
params/params/decoder/compose_15/mudd_postnorm_15/scale (20

In [4]:
%load_ext autoreload
%autoreload 2

import sys
    
sys.path.append('/home/lishengping/project/dreamily-v3.5-deploy/app')

import torch
import sentencepiece as spm
from configuration_dcformer import DCMuddDMoEConfig as ModelConfig # 
# 记得注释掉 from env import args，把from .model...形式的import改为from model...，不然会报错
from modeling_dcformer import DCFormer
# tokenizer = spm.SentencePieceProcessor(model_file=TOKENIZER_PATH)

torch_config = ModelConfig()
torch_config.torch_dtype = torch.bfloat16

model = DCFormer(torch_config)

# TOKENIZER_PATH = '/home/lishengping/tokenizer/spm_model_70000vocab_55G_bpe_character_coverage0.99999.extended_special.model'
# torch_params = torch.load('torch_params.bin') # torch.save之后的参数
IncompatibleKeys = model.load_state_dict(torch_params, strict=False)
for m in IncompatibleKeys.missing_keys:
    if 'layers.0' in m:
        print(m)
        
model.eval()
# model.half()
model.bfloat16()
# model.float()

with torch.device(model.device):
    model.setup_caches(max_batch_size=1, set_kv_cache=True)



rope_type: maxtext
layers.0.attention.dyn_w_proj.dw_m
layers.0.attention.dyn_w_proj.qkw_m
layers.0.attention.dyn_w_proj.qkw_bias


In [5]:
model.save_pretrained('v3.5mini-moe-S160000_bf16', safe_serialization=False)


## 下面表示测试代码

In [121]:
import os
import time
import argparse
import socket
import random
from collections import defaultdict

os.environ["JAX_PLATFORMS"] = "cpu"

import tensorflow as tf
import jax
import numpy as np


def _parse_function(example_proto):
    feature_desc = {key: tf.io.VarLenFeature(tf.int64) for key in task_features}
    example = tf.io.parse_single_example(example_proto, feature_desc)
    for name in list(example.keys()):
        t = example[name]
        if t.dtype == tf.int64:
            t = tf.cast(t, dtype=tf.int32)
        example[name] = tf.sparse.to_dense(t, default_value=0)[: seq_len]
        print(f'example[name]: {example[name]}')
    return example

task_features = {'input_ids': None}
train_seed = 1234
num_infeed_hosts = 1
shuffle_buffer_size = None
pad_id = 0
batch_size = 1
seq_len = 4097

p = 'gs://newproject-1-llm_base_models_europe-west4/data/xiaomeng/v3.5mini/unigram_tfids0506/validation/R000.000000'
fname = [p]
tf.random.set_seed(train_seed)
ds = tf.data.Dataset.from_tensor_slices(fname)
ds = ds.apply(tf.data.TFRecordDataset)
ds = ds.shard(num_infeed_hosts, 0)
ds = ds.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE)
if shuffle_buffer_size is not None:
    ds = ds.shuffle(buffer_size=shuffle_buffer_size)
padded_shapes = {key: seq_len for key in task_features}
padding_values = {key: pad_id for key in task_features}
ds = ds.padded_batch(
    batch_size=np.prod(batch_size),
    padded_shapes=padded_shapes,
    padding_values=padding_values,
    drop_remainder=True,
)
ds_iter = ds.as_numpy_iterator()

example[name]: Tensor("strided_slice:0", shape=(None,), dtype=int32)


In [122]:
length = 256
torch_bf16_losses = []
for i in range(20):
    a = next(ds_iter)
    batch_indexes = torch.tensor([0])
    inputs = torch.from_numpy(a['input_ids'][:, :length]).long()
    labels = torch.from_numpy(a['input_ids'][:, 1:length+1]).long()
    input_pos = torch.arange(length).unsqueeze(0)
    
    with torch.no_grad():
        output = model.forward(inputs, 
                               input_pos, 
                               batch_indexes=batch_indexes,
                               kvshift_cache_poss=torch.tensor([[0, -1]]))
    
    loss_fn = torch.nn.CrossEntropyLoss(reduction='none')
    
    loss = loss_fn(output.logits.view(-1, 70000).float(), labels.view(-1))
    loss = round(loss.mean().item(), 4)
    print(f'loss: {loss}')
    torch_bf16_losses.append(loss)

loss: 1.6847
loss: 2.8498
loss: 2.6098
loss: 2.7121
loss: 3.2024
loss: 2.5705
loss: 2.6805
loss: 2.6454
loss: 2.7186
loss: 2.6827
loss: 2.7481
loss: 2.4602
loss: 2.8353
loss: 2.7139
loss: 2.9138
loss: 2.6166
loss: 3.0291
loss: 3.3991
loss: 2.6877
loss: 2.4443


In [1]:
# jax bf16
jax_bf16 = '''loss: 1.6829
loss: 2.8398
loss: 2.6041
loss: 2.7052
loss: 3.2018
loss: 2.5704
loss: 2.6778
loss: 2.6341
loss: 2.7242
loss: 2.6773
loss: 2.7471
loss: 2.4557
loss: 2.8426
loss: 2.7089
loss: 2.9181
loss: 2.6191
loss: 3.0235'''
jax_bf16 = [float(l.split(':')[-1].strip()) for l in jax_bf16.split('\n')]
sum(jax_bf16) / len(jax_bf16)

2.6842705882352944

In [3]:
# fp16
fp16 = '''loss: 1.6844
loss: 2.8463
loss: 2.6096
loss: 2.7075
loss: 3.2011
loss: 2.5706
loss: 2.6799
loss: 2.6418
loss: 2.7193
loss: 2.6686
loss: 2.75
loss: 2.4499
loss: 2.8391
loss: 2.7215
loss: 2.9187
loss: 2.611
loss: 3.0318
loss: 3.4008
loss: 2.6905
loss: 2.4439'''

# bf16
bf16 = '''loss: 1.6847
loss: 2.8498
loss: 2.6098
loss: 2.7121
loss: 3.2024
loss: 2.5705
loss: 2.6805
loss: 2.6454
loss: 2.7186
loss: 2.6827
loss: 2.7481
loss: 2.4602
loss: 2.8353
loss: 2.7139
loss: 2.9138
loss: 2.6166
loss: 3.0291
loss: 3.3991
loss: 2.6877
loss: 2.4443'''
# fp32
fp32 = '''loss: 1.6841
loss: 2.8377
loss: 2.6093
loss: 2.7057
loss: 3.2037
loss: 2.5709
loss: 2.6784
loss: 2.6406
loss: 2.7157
loss: 2.6771
loss: 2.7519
loss: 2.4469
loss: 2.8421
loss: 2.7318
loss: 2.9159
loss: 2.6128
loss: 3.0295
loss: 3.4022
loss: 2.6902
loss: 2.4435'''


In [4]:
fp16_ = [float(l.split(':')[-1].strip()) for l in fp16.split('\n')][:len(jax_bf16)]
print(sum(fp16_) / len(fp16_))

bfp16_ = [float(l.split(':')[-1].strip()) for l in bf16.split('\n')][:len(jax_bf16)]
print(sum(bfp16_) / len(bfp16_))

fp32_ = [float(l.split(':')[-1].strip()) for l in fp32.split('\n')][:len(jax_bf16)]
print(sum(fp32_) / len(fp32_))

2.6853588235294112
2.6866764705882353
2.6855352941176465


In [ ]:
print(fp16_)
print(bfp16_)
print(fp32_)

[1.6844, 2.8463, 2.6096, 2.7075, 3.2011, 2.5706, 2.6799, 2.6418, 2.7193, 2.6686, 2.75, 2.4499, 2.8391, 2.7215, 2.9187]
[1.6847, 2.8498, 2.6098, 2.7121, 3.2024, 2.5705, 2.6805, 2.6454, 2.7186, 2.6827, 2.7481, 2.4602, 2.8353, 2.7139, 2.9138]
[1.6841, 2.8377, 2.6093, 2.7057, 3.2037, 2.5709, 2.6784, 2.6406, 2.7157, 2.6771, 2.7519, 2.4469, 2.8421, 2.7318, 2.9159]
